# Antarctic Iceberg Drift Physics Engine — Demo

A walkthrough of the physics-first engine in `src/iceberg_model/`: buoyancy,
forces, a full synthetic simulation, and where the live-data pipeline plugs in.

This notebook uses **synthetic in-memory data only** (no external files, no
network access) — the same pattern as `examples/run_real_data.py`. For
pulling real ERA5/AMSR2/Copernicus Marine/BedMachine data, see
`examples/fetch_and_run_live.py` and `docs/data_pipeline.md`.


In [ ]:
import sys
from pathlib import Path

# Make src/ importable when running this notebook from notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from iceberg_model.config.physics_config import PhysicsConfig
from iceberg_model.state.iceberg_state import IcebergState
from iceberg_model.physics.buoyancy import calculate_submerged_depth, calculate_freeboard
from iceberg_model.physics.forces import iceberg_mass, calculate_total_acceleration
from iceberg_model.data.environmental_dataset import EnvironmentalDataset
from iceberg_model.data.temporal_loader import TemporalField
from iceberg_model.simulation.simulator import Simulator
from iceberg_model.physics.melting import MeltRatesConfig
from iceberg_model.validation.physical_checks import check_state

config = PhysicsConfig()
config


## 1. A single iceberg: mass, buoyancy, freeboard

In [ ]:
state = IcebergState(x=0.0, y=0.0, u=0.0, v=0.0, L=800.0, W=400.0, H=150.0)

mass = iceberg_mass(state, config)
D = calculate_submerged_depth(state, config)
freeboard = calculate_freeboard(state, config)

print(f"Mass:            {mass:.3e} kg")
print(f"Keel depth:      {D:.1f} m")
print(f"Freeboard:       {freeboard:.1f} m")
print(f"Check D+freeboard == H: {D + freeboard:.4f} == {state.H}")


## 2. A synthetic environment

Real deployments build this via `data/live_pipeline.py::LiveEnvironmentBuilder`
(ERA5 wind, AMSR2 sea ice, Copernicus Marine currents, BedMachine bathymetry)
or from local GeoTIFF/NetCDF files via `data/geotiff_reader.py` /
`data/netcdf_loader.py` + `data/reprojection.py`. Here we build one by hand
so the notebook runs standalone.

In [ ]:
def build_synthetic_environment(grid_size=50, resolution_m=2000.0):
    shape = (grid_size, grid_size)
    half_extent = grid_size * resolution_m / 2
    transform = (resolution_m, 0.0, -half_extent, 0.0, -resolution_m, half_extent)

    ocean_u = np.full(shape, 0.3)
    ocean_v = np.zeros(shape)
    wind_u = np.zeros(shape)
    wind_v = np.full(shape, -4.0)
    sic = np.clip(1.0 - np.linspace(0, 1, grid_size)[None, :].repeat(grid_size, axis=0), 0, 1)
    sst = np.full(shape, 1.5)
    bathymetry = np.full(shape, 500.0)
    latitude = np.full(shape, -65.0)

    return EnvironmentalDataset(
        transform=transform,
        temporal_fields={
            "ocean_u": TemporalField(timestamps=[0.0, 1e6], fields=[ocean_u, ocean_u]),
            "ocean_v": TemporalField(timestamps=[0.0, 1e6], fields=[ocean_v, ocean_v]),
            "wind_u": TemporalField(timestamps=[0.0, 1e6], fields=[wind_u, wind_u]),
            "wind_v": TemporalField(timestamps=[0.0, 1e6], fields=[wind_v, wind_v]),
            "sea_ice_concentration": TemporalField(timestamps=[0.0, 1e6], fields=[sic, sic]),
            "sea_surface_temperature": TemporalField(timestamps=[0.0, 1e6], fields=[sst, sst]),
        },
        static_fields={"bathymetry": bathymetry},
        latitude_field=latitude,
    )

environment = build_synthetic_environment()
sample = environment.sample_environment(x=0.0, y=0.0, t=0.0, config=config)
sample


## 3. Force breakdown at a single instant

In [ ]:
breakdown = calculate_total_acceleration(state, sample, config)

forces = {
    "ocean": breakdown.ocean,
    "air": breakdown.air,
    "coriolis": breakdown.coriolis,
    "pressure": breakdown.pressure,
    "sea_ice": breakdown.sea_ice,
    "grounding": breakdown.grounding,
}

fig, ax = plt.subplots(figsize=(6, 6))
for name, vec in forces.items():
    ax.arrow(0, 0, vec[0] * 1e6, vec[1] * 1e6, head_width=0.05, label=f"{name} (x1e6)")
ax.arrow(0, 0, breakdown.total[0] * 1e6, breakdown.total[1] * 1e6, head_width=0.08,
         color="black", linewidth=2, label="total (x1e6)")
ax.set_xlabel("a_x [m/s^2] x 1e6")
ax.set_ylabel("a_y [m/s^2] x 1e6")
ax.set_title("Per-mechanism acceleration breakdown")
ax.legend(loc="upper left", fontsize=8)
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
plt.show()


## 4. Run a full simulation

In [ ]:
simulator = Simulator(config=config, environment=environment, melt_rates_config=MeltRatesConfig())

initial_state = IcebergState(x=0.0, y=0.0, u=0.0, v=0.0, L=800.0, W=400.0, H=150.0)
result = simulator.run(initial_state, t_end=48 * 3600.0)  # 48 hours

times, states = result.concatenated()
print(f"Simulated {len(times)} steps, final mode: {result.final_mode.value}")
print(f"Events: {result.event_log.events}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(states[:, 0] / 1000, states[:, 1] / 1000, marker="o", markersize=2)
axes[0].set_xlabel("x [km]")
axes[0].set_ylabel("y [km]")
axes[0].set_title("Iceberg trajectory (EPSG:3031)")
axes[0].set_aspect("equal")

hours = times / 3600
axes[1].plot(hours, states[:, 4], label="L")
axes[1].plot(hours, states[:, 5], label="W")
axes[1].plot(hours, states[:, 6], label="H")
axes[1].set_xlabel("time [hours]")
axes[1].set_ylabel("dimension [m]")
axes[1].set_title("Geometry over time (melting)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Physical sanity check on the final state

In [ ]:
final_state = IcebergState.from_vector(states[-1], time=times[-1], mode=result.final_mode)
report = check_state(final_state, config, water_depth_m=500.0)
print(f"ok={report.ok}")
print(f"violations={report.violations}")


## 6. Where the live-data pipeline plugs in

Everything above used `build_synthetic_environment()`. To run against real
data instead, swap that call for:

```python
from iceberg_model.data.fetchers.common import BoundingBox, TimeRange
from iceberg_model.data.live_pipeline import LiveEnvironmentBuilder
from iceberg_model.data.reprojection import build_common_grid

grid = build_common_grid(crs="EPSG:3031", resolution_m=5000.0,
                          bounds=(-2_000_000, -2_000_000, 2_000_000, 2_000_000))
bbox = BoundingBox(min_lon=-60, min_lat=-66, max_lon=-55, max_lat=-63)
time_range = TimeRange(start=..., end=...)

environment = LiveEnvironmentBuilder(grid=grid).build(bbox, time_range)
```

This requires `pip install -e ".[live-data]"` and free credentials for
Copernicus CDS, NASA Earthdata, and Copernicus Marine — see
`docs/data_pipeline.md` and `.env.example`. See `examples/fetch_and_run_live.py`
for the full runnable script.
